# 14 · Geographic building scenes with MapLibre

**Question:** How can a 3D map preserve geographic position and honest building heights?

**Runtime:** Python / Pyodide in JupyterLite. Choose **Run → Run All Cells**.
Each notebook is independent; start with a fresh kernel. Initial runtime/package downloads need internet.
The initial baseline uses **synthetic educational fixtures** near a Gaborone-area anchor.
Extended investigations explicitly identify bundled real OSM, Sentinel or terrain extracts where used;
these do not validate the synthetic population, mobility, exposure or scenario assumptions.

**Astra experiment:** Ask Astra to audit a 3D scene against its GeoJSON, then have Codex add a visual control with explicit data invariants.
Run the numerical baseline first, then give Astra the exported evidence and the prompt below.
The baseline does not call or run GPT-6 Astra. No API key is needed.


In [ ]:
import sys, json, math, time
from pathlib import Path
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from twin import *
style()
print("Runtime:", sys.platform, "| data: synthetic | seed: 42")


## From footprint to extrusion · 30 minutes
**Learn:** create WGS84 GeoJSON, distinguish local coordinates from geographic coordinates, and use a
MapLibre `fill-extrusion` layer. Horizontal coordinates are longitude/latitude; heights are metres above
an assumed flat ground plane. Floor height is invented, not measured. Change `floor_height_m` and rerun.
MapLibre projects the GeoJSON for display; do not put local east/north metres directly into its coordinates.
**Predict:** should doubling floor height change population or footprint area?


In [ ]:
from scenes import building_features, export_scene, scene_frame
c=city(); pop=allocate(c['capacity'],1200); floor_height_m=3.
buildings=building_features(c,pop,floor_height_m)
nodes,graph=network()
roads=dict(type='FeatureCollection',features=[dict(type='Feature',properties={},geometry=dict(type='LineString',
    coordinates=[[float(v) for v in lonlat(*nodes[u])],[float(v) for v in lonlat(*nodes[v])]]))
    for u in graph for v,w in graph[u] if u<v])
assert sum(f['properties']['population'] for f in buildings['features'])==1200
assert all(f['geometry']['coordinates'][0][0]==f['geometry']['coordinates'][0][-1] for f in buildings['features'])
heights=np.array([f['properties']['height_m'] for f in buildings['features']])
assert np.allclose(heights,c['floors']*floor_height_m)
center=[float(v) for v in lonlat(1000,1000)]
payload=dict(buildings=buildings,roads=roads,center=center,synthetic=True,vertical_units='metres above flat ground')
export_json('14_buildings.geojson',buildings)
print('Buildings:',len(heights),'| physical heights:',sorted(set(heights)),'m | residents:',pop.sum())


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,5))
dots=axes[0].scatter(*c['xy'].T,c=heights,s=25,cmap='viridis'); map_axes(axes[0],'Plan-view height audit')
fig.colorbar(dots,ax=axes[0],label='Assumed height (m)')
axes[1].hist(heights,bins=np.arange(.5,4.5)*floor_height_m,color='#087f8c')
axes[1].set(xlabel='Assumed building height (m)',ylabel='Buildings',title='Unexaggerated height distribution')
plt.tight_layout(); plt.show()


## Use and share the interactive scene
The scene appears below in a trusted notebook output. If your notebook viewer blocks the iframe,
download the HTML from `exports/` and open it in a browser. All data are embedded, but the pinned
JavaScript/CSS renderer still downloads from a CDN; this is not fully offline. WebGL must be available.
The static plot and numerical outputs remain usable when the renderer cannot load.

Camera rotation and vertical exaggeration change the view, not the underlying data. Keep a plan view
beside a perspective view to detect occlusion. Download the notebook, scene and evidence JSON before
clearing browser storage. Do not label synthetic geometry as observed Gaborone buildings.


In [ ]:
document=export_scene('14_maplibre_scene.html','maplibre',payload)
display(scene_frame(document,'MapLibre synthetic buildings in 3D'))


## Astra interpretation and a Codex implementation task
**Astra:** send `14_buildings.geojson` plus a screenshot at 1× and 5× exaggeration.
Ask: “Which differences come from the camera or exaggeration? Which heights are supported by the data?
List the coordinate order, height units and assumptions. Do not infer occupancy or neighborhood deprivation
from appearance.” Score numerical claims against the GeoJSON, and mark screenshot-only impressions separately.

**Codex:** “Add a minimum-floor filter to `content/scene_templates/maplibre.html`. Show the visible count,
preserve IDs and raw heights, and keep the accessible table consistent with the filter. Test zero matches
and restoring all buildings. Do not rescale heights to make the scene look more dramatic.”

**Acceptance:** 144 original IDs; population total 1,200 before filtering; extrusion equals floors × floor height ×
display exaggeration; changing camera leaves data unchanged. Copy the exact prompt, response, model/date and
screenshots into your learning record. The initial notebook contains no measured Astra response.

Reference: [MapLibre 3D buildings](https://maplibre.org/maplibre-gl-js/docs/examples/display-buildings-in-3d/).
Our scene uses original GeoJSON and an empty style rather than the example's external tile source.


## Extended investigation: Import real footprints and preserve unknown heights

Import the bundled OSM extract with source IDs, timestamps and license. None of these 125 buildings reports a numeric height; the map therefore keeps them flat and labels height as unknown. A separate chart shows how assumed heights change hypothetical volume, explicitly distinguishing assumptions from observations. The interactive building scene now includes the requested minimum-floor filter, visible counts and a synchronized table.

**Read the concept map first:** identify the input, method, evidence and limit. Predict the spatial pattern before running the comparison.


In [ ]:
from extension_tools import concept_map, export_investigation
concept_map('Import real footprints and preserve unknown heights', ['OSM IDs + height tags', 'Audit / filter / preserve nulls', 'Real map + assumption range', 'Missing height stays unknown'])
plt.show()


In [ ]:
from extension_tools import load_data, audit_footprints
from scenes import export_scene, scene_frame
from copy import deepcopy
ex_real=load_data('gaborone_buildings.geojson'); ex_audit=audit_footprints(ex_real)
ex_features=[]; ex_areas=[]
for f,r in zip(ex_real['features'],ex_audit):
    if r['issues']: continue
    feature=deepcopy(f); p=feature['properties']; ex_areas.append(r['area_m2'])
    p.update(id=f['id'],floors=p['levels'],population=None,residential=False,height_known=p['height_m'] is not None)
    ex_features.append(feature)
ex_known=sum(f['properties']['height_known'] for f in ex_features)
ex_coordinates=np.concatenate([np.array(f['geometry']['coordinates'][0]) for f in ex_features])
fig,axes=plt.subplots(1,2,figsize=(12,5))
for f in ex_features:
    ring=np.array(f['geometry']['coordinates'][0]); axes[0].fill(*np.array(local_xy(*ring.T)),color='#8c9ca5',alpha=.65)
axes[0].set(aspect='equal',xlabel='Local east (m)',ylabel='Local north (m)',title='Real footprints; height unavailable')
ex_assumed=[3,6,9]; ex_volume=[float(sum(ex_areas)*h) for h in ex_assumed]
axes[1].bar(['Assume 3 m','Assume 6 m','Assume 9 m'],ex_volume,color='#b88741'); axes[1].set(ylabel='Hypothetical total volume (m³)',title='Height assumptions, NOT observed volume')
fig.text(.01,.01,'© OpenStreetMap contributors · ODbL 1.0 · openstreetmap.org/copyright',fontsize=9); fig.tight_layout(rect=[0,.05,1,1]); extension_figure=fig
ex_payload=dict(buildings=dict(type='FeatureCollection',features=ex_features),roads=dict(type='FeatureCollection',features=[]),center=ex_coordinates.mean(axis=0).tolist(),zoom=17.5,
    title='Real footprints, unknown heights',description='OSM geometry is observed mapping evidence; absent heights remain unknown and render flat.',attribution='© OpenStreetMap contributors · ODbL 1.0 · openstreetmap.org/copyright',synthetic=False)
ex_document=export_scene('14_real_footprints.html','maplibre',ex_payload); display(scene_frame(ex_document,'Real OSM footprint audit with unknown heights'))
assert ex_known==sum(f['properties']['height_m'] is not None for f in ex_features)
extension_evidence=dict(provenance=ex_real['provenance'],feature_count=len(ex_features),reported_height_count=ex_known,total_footprint_m2=float(sum(ex_areas)),assumed_heights_m=ex_assumed)
extension_expected=dict(measured_height_count=ex_known,can_infer_real_volume=False)
extension_task='Return measured_height_count (reported numeric height fields) and can_infer_real_volume. Missing tags must not become fabricated measurements.'


In [ ]:
extension_prompt,extension_reference=export_investigation('14',extension_evidence,extension_expected,extension_task,extension_figure)
plt.show()
print('Send only 14_extension_prompt.json and optionally 14_extension_map.png to Astra. Keep the reference in this kernel.')


### Ask, check and explain the spatial result
Download `14_extension_prompt.json` and `14_extension_map.png`. Send the evidence in a fresh Astra
session with the exact task and output fields. For a separate image-only experiment, supply only the image
and task wording, record that condition, and allow clarification if the image does not contain enough information.
Never send the reference or this complete notebook for a blind trial.

Paste the unedited response below or upload a UTF-8 response file. The default is deliberately empty:
**no model trial has run**. After recording a real answer, set `extension_source='MODEL'` and complete the metadata.
The numeric tolerance is 0.01 in each requested field's declared units; labels and booleans require exact agreement.
Keep that rule fixed before examining a response. Inspect the explanation separately for unsupported claims.

**Visual reflection:** point to one map pattern, cite the numerical check that supports it, and name one thing
the visual cannot establish. Re-run one parameter change and preserve both maps. The implementation above
is available for inspection and modification; it is not a measured claim about model accuracy.


In [ ]:
from experiments import grade_answer
extension_source='NOT RUN'
extension_model=''; extension_date=''; extension_interface=''; extension_condition='text-and-map'
extension_response_file=''; extension_response_text=''
if extension_response_file:
    extension_response_text=Path(extension_response_file).read_text(encoding='utf-8')
if extension_source=='MODEL':
    assert extension_model and extension_date and extension_interface and extension_response_text, 'Record the actual model, date, interface and response'
extension_tolerances={key:.01 for key,value in extension_reference.items() if type(value) in (int,float)}
extension_grade=grade_answer(extension_response_text,extension_reference,extension_tolerances) if extension_response_text else dict(status='NOT RUN')
display(extension_grade)
export_json('14_extension_trial.json',dict(source=extension_source,model=extension_model,date=extension_date,
    interface=extension_interface,condition=extension_condition,prompt=extension_prompt,response_text=extension_response_text,
    tolerances=extension_tolerances,reference=extension_reference,grade=extension_grade))


## Save the investigation
Download the edited notebook, evidence, map and trial record from `exports/`. Browser storage does not
commit changes to GitHub. Keep data attribution and the distinction between observed data, synthetic
assumptions and actual model responses when sharing. The extended code and its checks above are part of
this notebook; use them as the starting point for your next controlled comparison.
